In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline

In [3]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR


In [4]:
chemin_fichier = "C:/Users/zizou/OneDrive/Desktop/stage 3ème/day 2/csvfiles/dataframefinale.csv"
df = pd.read_csv(chemin_fichier, sep=';')

In [5]:
df

,dataloadingdate,Jour,Mois,NumeroSemaine,Trimestre,JourSemaineNum,JourSemaine,ISIN,Libellé,Nombre de Titres,Montant,Echéance,Taux
0,01/07/2025,1,7,27,3,1,Tuesday,TN0008000739,"BTA 7,4% Fevrier 2030",1459.0,1.500,62.0,7.60
1,01/07/2025,1,7,27,3,1,Tuesday,TN0008000739,"BTA 7,4% Fevrier 2030",291.0,0.300,183.0,7.50
2,01/07/2025,1,7,27,3,1,Tuesday,TNMCPXLL1EE2,EMP NAT 2023 T4 CB TV,50000.0,5.000,13.0,8.60
3,01/07/2025,1,7,27,3,1,Tuesday,TNMCPXLL1EE2,EMP NAT 2023 T4 CB TV,10000.0,1.000,7.0,8.60
4,01/07/2025,1,7,27,3,1,Tuesday,TN0008000606,"BTA 6,7% Avril 2028",5660.0,5.742,31.0,9.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19967,31/12/2024,31,12,1,4,1,Tuesday,TN0008000812,"BTA 7,5% 13/12/2028",215.0,0.200,50.0,9.00
19968,31/12/2024,31,12,1,4,1,Tuesday,TNX0K9990B08,EMP NAT 2024 T2 CB TF,5243.0,0.556,15.0,7.49
19969,31/12/2024,31,12,1,4,1,Tuesday,TNX0K9990B08,EMP NAT 2024 T2 CB TF,28288.0,3.000,31.0,8.99
19970,31/12/2024,31,12,1,4,1,Tuesday,TNX0K9990B08,EMP NAT 2024 T2 CB TF,48701.0,5.165,31.0,8.99


In [6]:
features = df.drop(columns=["dataloadingdate", "Jour", "Mois", "NumeroSemaine", "Trimestre", "JourSemaineNum", "JourSemaine", "Montant"])
target = df["Montant"]

X = features
y = target

In [7]:
# 📊 Fonction générique pour appliquer les 4 étapes avec un modèle
def evaluer_modele(nom, modele, params=None):
    print(f"\n=== Modèle : {nom} ===")

    # 1. Sans traitement
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    modele.fit(X_train, y_train)
    pred = modele.predict(X_test)
    rmse1 = np.sqrt(mean_squared_error(y_test, pred))
    print(f"[1] RMSE sans traitement : {rmse1:.4f}")

    # 2. Avec normalisation
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    modele.fit(X_train, y_train)
    pred = modele.predict(X_test)
    rmse2 = np.sqrt(mean_squared_error(y_test, pred))
    print(f"[2] RMSE avec normalisation : {rmse2:.4f}")

    # 3. Sélection de variables
    selector = SelectKBest(score_func=f_regression, k=min(5, X.shape[1]))
    X_selected = selector.fit_transform(X_scaled, y)
    X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)
    modele.fit(X_train, y_train)
    pred = modele.predict(X_test)
    rmse3 = np.sqrt(mean_squared_error(y_test, pred))
    print(f"[3] RMSE après feature selection : {rmse3:.4f}")

    # 4. Hyperparameter tuning si params fournis
    if params:
        grid = GridSearchCV(estimator=modele, param_grid=params, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1)
        grid.fit(X_selected, y)
        best_model = grid.best_estimator_
        pred = best_model.predict(X_selected)
        rmse4 = np.sqrt(mean_squared_error(y, pred))
        print(f"[4] RMSE après tuning : {rmse4:.4f} (meilleur params: {grid.best_params_})")
    else:
        print("[4] Tuning ignoré : aucun paramètre fourni")

In [9]:
evaluer_modele("Linear Regression", LinearRegression())


=== Modèle : Linear Regression ===


ValueError: could not convert string to float: 'TN9092FJVKK8'